# Moodify — Master Google Colab (Six-Emotion)

End-to-end workflow: data → six-class model → emotion-specific metadata/content relevance ranking → Streamlit → health check → temporary tunnel.

Song emotion is **not ground truth**: catalogue relevance is inferred from available metadata/content signals.

In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil, subprocess, time, urllib.request, re, platform, stat
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
BASE = Path("/content") if Path("/content").exists() and os.access("/content", os.W_OK) else Path.cwd()
WORKSPACE = BASE / "moodify_workspace"; WORKSPACE.mkdir(parents=True, exist_ok=True)
def find_project_root(base):
    candidates = [p for p in [base] + [x for x in base.rglob("*") if x.is_dir()] if (p/"app.py").exists() and (p/"src").exists()]
    return sorted(candidates, key=lambda p: len(p.parts))[0] if candidates else None
PROJECT_ROOT = find_project_root(BASE)
if PROJECT_ROOT is None: PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB:
    uploaded = files.upload(); zip_files = [Path(n) for n in uploaded if n.lower().endswith(".zip")]
    if not zip_files: raise FileNotFoundError("Upload the Moodify six-emotion ZIP")
    extract_dir = WORKSPACE/"project"; shutil.rmtree(extract_dir, ignore_errors=True); extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_files[0]) as z: z.extractall(extract_dir)
    PROJECT_ROOT = find_project_root(extract_dir)
if PROJECT_ROOT is None: raise FileNotFoundError("Could not find Moodify project root")
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT)); print(PROJECT_ROOT)

In [ ]:
req = PROJECT_ROOT / "requirements.txt"
if req.exists(): subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)
import joblib, pandas as pd, json
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.emotion_data import EMOTIONS, EMOTION_NAMES
from src.modeling import build_final_model
from src.labeling import add_emotion_relevance
from src.recommender import predict_emotion, emotion_scores, recommend_by_emotion
required = ["emotion_train.csv", "emotion_validation.csv", "emotion_test.csv", "src/recommender.py", "data/processed/labelled_catalogue.csv"]
missing = [p for p in required if not (PROJECT_ROOT/p).exists()]
if missing: raise FileNotFoundError(missing)
print("Required files verified", EMOTIONS)

## 1. Load the original six-class emotion data

In [ ]:
train = pd.read_csv(PROJECT_ROOT/"emotion_train.csv"); val = pd.read_csv(PROJECT_ROOT/"emotion_validation.csv"); test = pd.read_csv(PROJECT_ROOT/"emotion_test.csv")
assert train.shape == (16000, 2) and val.shape == (2000, 2) and test.shape == (2000, 2)
assert sorted(train["label"].unique()) == list(range(6))
for name, d in [("train", train), ("validation", val), ("test", test)]: print(name, d.shape, d.label.value_counts().sort_index().to_dict())


## 2. Train and evaluate the genuine six-class model

In [ ]:
model = build_final_model(); model.fit(train.text, train.label.map(EMOTION_NAMES))
val_pred = model.predict(val.text); print("Validation accuracy:", round(accuracy_score(val.label.map(EMOTION_NAMES), val_pred), 4)); print("Validation macro-F1:", round(f1_score(val.label.map(EMOTION_NAMES), val_pred, average="macro"), 4))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42); cv_scores = cross_val_score(build_final_model(), train.text, train.label.map(EMOTION_NAMES), cv=cv, scoring="f1_macro", n_jobs=-1); print("5-fold macro-F1:", round(cv_scores.mean(), 4), "+/-", round(cv_scores.std(), 4))


In [ ]:
final_model = build_final_model(); X = pd.concat([train.text, val.text], ignore_index=True); y = pd.concat([train.label.map(EMOTION_NAMES), val.label.map(EMOTION_NAMES)], ignore_index=True); final_model.fit(X, y)
test_y = test.label.map(EMOTION_NAMES); test_pred = final_model.predict(test.text)
report = classification_report(test_y, test_pred, labels=list(EMOTIONS), output_dict=True, zero_division=0)
metrics = {"accuracy": float(accuracy_score(test_y, test_pred)), "balanced_accuracy": float(balanced_accuracy_score(test_y, test_pred)), "macro_f1": float(f1_score(test_y, test_pred, average="macro")), "classification_report": report, "labels": list(EMOTIONS), "confusion_matrix": confusion_matrix(test_y, test_pred, labels=list(EMOTIONS)).tolist(), "evaluation_note": "Six-class Linear SVM evaluated on the held-out 2,000-row test split; train+validation used for final fit."}
print("Test accuracy:", round(metrics["accuracy"], 4)); print("Test balanced accuracy:", round(metrics["balanced_accuracy"], 4)); print("Test macro-F1:", round(metrics["macro_f1"], 4)); print(classification_report(test_y, test_pred, labels=list(EMOTIONS), digits=4)); display(pd.DataFrame(confusion_matrix(test_y, test_pred, labels=list(EMOTIONS)), index=EMOTIONS, columns=EMOTIONS))
(PROJECT_ROOT/"emotion_model_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")


## 3. Prepare metadata/content-based emotion relevance and rank recommendations

In [ ]:
from src.data import prepare_catalogue
raw_catalogue = pd.read_csv(PROJECT_ROOT/"data/raw/spotify_metadata_catalogue.csv")
catalogue = add_emotion_relevance(prepare_catalogue(raw_catalogue))
catalogue.to_csv(PROJECT_ROOT/"data/processed/labelled_catalogue.csv", index=False)
assert len(catalogue) == 2878
assert all(f"emotion_{e}" in catalogue.columns for e in EMOTIONS)
print("Catalogue emotion labels:", catalogue.emotion_label.value_counts(dropna=False).to_dict())
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2, max_features=12000, sublinear_tf=True); song_matrix = vectorizer.fit_transform(catalogue.recommendation_text.fillna(catalogue.text).fillna(""))
for emotion in EMOTIONS:
    recs = recommend_by_emotion(catalogue, emotion, n=5, query=f"I feel {emotion}", vectorizer=vectorizer, song_matrix=song_matrix)
    assert len(recs) == 5 and "rank_score" in recs.columns and recs["rank_score"].is_monotonic_decreasing
    print(emotion, len(recs), recs.iloc[0].song_name)


In [ ]:
MODEL_PATH = PROJECT_ROOT/"models/moodify_model.joblib"; MODEL_PATH.parent.mkdir(parents=True, exist_ok=True); joblib.dump(final_model, MODEL_PATH); loaded = joblib.load(MODEL_PATH)
assert set(loaded.classes_) == set(EMOTIONS), loaded.classes_
assert set(predict_emotion(loaded, f"I feel {e}") for e in EMOTIONS).issubset(set(EMOTIONS))
print("Saved six-class artifact:", MODEL_PATH)

## 4. Streamlit deployment and health check

The following cells start the unchanged polished UI with the updated six-emotion backend and create the existing temporary Quick Tunnel.

In [ ]:
APP_PATH = PROJECT_ROOT/"app.py"; LOG_PATH = PROJECT_ROOT/"streamlit.log"
if "STREAMLIT_PROCESS" in globals() and STREAMLIT_PROCESS.poll() is None: STREAMLIT_PROCESS.terminate(); time.sleep(2)
log_file = open(LOG_PATH, "w", encoding="utf-8")
STREAMLIT_PROCESS = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP_PATH), "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true", "--browser.gatherUsageStats=false"], cwd=PROJECT_ROOT, stdout=log_file, stderr=subprocess.STDOUT)
healthy = False
for _ in range(60):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2) as r:
            if r.status == 200: healthy = True; break
    except Exception: pass
if not healthy: raise RuntimeError(LOG_PATH.read_text(errors="ignore")[-6000:])
print("Streamlit healthy: http://127.0.0.1:8501")

In [ ]:
CLOUDFLARED = WORKSPACE / "cloudflared"
if not CLOUDFLARED.exists():
    asset = "cloudflared-linux-amd64" if platform.machine().lower() in ("x86_64", "amd64") else "cloudflared-linux-arm64"
    urllib.request.urlretrieve(f"https://github.com/cloudflare/cloudflared/releases/latest/download/{asset}", CLOUDFLARED); CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IEXEC)
TUNNEL_PROCESS = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for _ in range(60):
    line = TUNNEL_PROCESS.stdout.readline(); print(line.rstrip())
    if re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line): break


## Final checks

The production path contains no three-mood mapping. The model classes, selected UI emotion, metadata/content relevance score, TF-IDF similarity, and popularity signal are all six-emotion compatible. Song emotion remains an inferred metadata/content relevance signal, not ground truth.